# Reproducing Jackson, Cole, Voytek & Swann (2019) from OpenNeuro ds002778

**Paper.** Jackson N, Cole SR, Voytek B, Swann NC (2019). Characteristics of waveform shape in
Parkinson's disease detected with scalp electroencephalography. *eNeuro* 6(3)
ENEURO.0151-19.2019. [doi:10.1523/ENEURO.0151-19.2019](https://doi.org/10.1523/ENEURO.0151-19.2019).
The claims file was written from the version of record (Europe PMC PMC6553574: main text,
Methods, legends, Tables 1 to 3) and the paper's code, SwannLab/Jackson_2019 at commit a7c89f5.

**Dataset.** [OpenNeuro ds002778](https://openneuro.org/datasets/ds002778/versions/1.0.5),
"UC San Diego Resting State EEG Data from Patients with Parkinson's Disease", snapshot 1.0.5,
DOI 10.18112/openneuro.ds002778.v1.0.5. The paper (2019) predates the dataset (first snapshot
1.0.0, 2020-05-05) and cites no version. 1.0.5 is the most recent published snapshot. Its
CHANGES list fixes to the README, citations, a medication entry of sub-pd22 (1.0.1) and Beck
scores (1.0.5); none changes the EEG files, so no earlier snapshot matches the paper better.
Files are read from the URLs the OpenNeuro API gives for this snapshot, into memory.

**What the dataset holds.** Resting EEG (BioSemi, 32 scalp electrodes and 8 external
channels, 512 Hz) of 15 patients off and on medication and 16 controls, with UPDRS and other
scores in per-session JSON files. It holds no artifact marks and no bad-channel marks, which
the paper's preprocessing used; the analysis here therefore rejects no samples and excludes no
electrode (the claims file's reading (b) for both).

| Claim | Tested here |
|---|---|
| fig2-sharpness-off-on, fig2-steepness-off-on, fig2-pac-off-on | tested |
| bipolar-off-on | tested |
| temporalis-f7f8-null, temporalis-t7t8-null | tested |
| fig2c-topography-sensorimotor | tested |
| offctrl-sharpness-null, offctrl-steepness, offctrl-pac-null, offctrl-pac-csd | tested |
| updrs-improvement | tested |
| fig1c-beta-alpha-peaks | tested |
| table1-alpha-shape-off-on, table1-alpha-pac-null, table2-alpha-offctrl | tested |
| pac-beta-larger-than-alpha | tested |
| fig3-sharpness-pac-correlation, fig3-steepness-pac-correlation | tested |
| fig4a-sensorimotor-q4, q4-patient-counts, fig4b-temporalis-variable-shape, controls-q4 | tested |
| table3-q4-effect-size | tested |

No claim is blocked. The measures, filters and statistics are in `waveshape.py`, whose
docstring says where each parameter comes from.

In [ ]:
import itertools
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import waveshape as ws

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
files = ws.listing()
recs = ws.sessions()
with ThreadPoolExecutor(4) as pool:
    results = list(pool.map(lambda r: ws.analyse(r.sub, r.ses), recs.itertuples()))
print(f"OpenNeuro {ws.DATASET_ID} snapshot {ws.SNAPSHOT}: {len(files)} files; "
      f"{len(results)} recordings analysed in {time.time() - started:.0f} s")

shape = pd.concat([r["shape"] for r in results], ignore_index=True)
pac = pd.concat([r["pac"] for r in results], ignore_index=True)
info = pd.DataFrame([r["info"] for r in results])
GROUP = {"off": "off", "on": "on", "hc": "control"}

## Data check

In [ ]:
kinds = files.index.str.replace(r"sub-(pd|hc)\d+", "sub-X", regex=True)
print("Files by kind in the snapshot:")
print(pd.Series(kinds).value_counts().sort_index().to_string())
bdf = files[files.index.str.endswith(".bdf")]
print(f"\nBDF files: {len(bdf)}, {bdf['size'].sum() / 1e9:.3f} GB; distinct checksums: {bdf['id'].nunique()}")
print("\nChecksum (id) of every file read:")
read = files[files.index.str.endswith((".bdf", "participants.tsv", "_beh.json", "_events.tsv",
                                        "_channels.tsv", "_eeg.json"))]
print(read["id"].to_string())

In [ ]:
participants = ws.read_tsv("participants.tsv")
participants["group"] = np.where(participants.participant_id.str.startswith("sub-pd"), "PD", "control")
print(participants.drop(columns=["rl_deficits", "notes"]).to_string(index=False))
summ = participants.groupby("group").agg(n=("participant_id", "size"),
                                         female=("gender", lambda g: (g.str.lower() == "f").sum()),
                                         age_mean=("age", "mean"), age_sd=("age", "std"))
print("\n", summ.round(1).to_string())
print("Paper (Methods, Participants): 15 PD, 8 female, 63.2 +/- 8.2 years; 16 controls, 9 female, 63.5 +/- 9.6 years.")
print("\nNotes column:")
print(participants.loc[participants.notes.notna(), ["participant_id", "notes"]].to_string(index=False))
print("\nRecordings per group:", recs.group.value_counts().to_dict())
pd_subs = sorted(recs[recs.ses == "off"]["sub"], key=lambda s: int(s[2:]))
print("Patients with both off and on:", len(set(recs[recs.ses == "off"]["sub"]) & set(recs[recs.ses == "on"]["sub"])))

The participant counts match the paper (15 patients, 16 controls). The table below checks the
recordings: sampling rate, duration against the sidecar, channel names, the units and size of
the signals, and the sessions the `notes` column says were exported from preprocessed EEGLAB
files.

In [ ]:
side = {}
for r in recs.itertuples():
    base = f"sub-{r.sub}/ses-{r.ses}/eeg/sub-{r.sub}_ses-{r.ses}_task-rest"
    side[(r.sub, r.ses)] = {"eeg": ws.read_json(base + "_eeg.json"),
                            "events": ws.read_tsv(base + "_events.tsv"),
                            "channels": ws.read_tsv(base + "_channels.tsv")}
rows = []
for i in info.itertuples():
    s = side[(i.sub, i.ses)]
    ch = s["channels"]
    sd = pd.Series(i.pre_sd_uv)
    rows.append({"sub": i.sub, "ses": i.ses, "sfreq": i.sfreq, "dur_s": i.n_times / i.sfreq,
                 "sidecar_dur_s": s["eeg"]["RecordingDuration"] + 1 / 512,
                 "n_chan": len(i.channels), "scalp_names_ok": i.channels[:32] == ws.SCALP,
                 "status_all_good": bool((ch.status == "good").all()),
                 "units": "/".join(sorted(ch.units.dropna().unique())), "highcut": ch.high_cutoff.iloc[0],
                 "raw_mean_abs_uv": round(i.raw_mean_abs_uv, 2),
                 "raw_max_abs_sum_uv": round(i.raw_sum_over_channels_uv, 3),
                 "pre_sd_min": round(sd.min(), 2), "pre_sd_median": round(sd.median(), 2),
                 "pre_sd_max": round(sd.max(), 1), "max_sd_chan": sd.idxmax(), "min_sd_chan": sd.idxmin()})
rec_tab = pd.DataFrame(rows)
print(rec_tab.to_string(index=False))
print("\nSampling rate values:", rec_tab.sfreq.unique().tolist())
print("Duration equals sidecar RecordingDuration + 1 sample in all:",
      bool(np.allclose(rec_tab.dur_s, rec_tab.sidecar_dur_s)))
print("Duration range (s):", rec_tab.dur_s.min(), "to", rec_tab.dur_s.max(), "(Methods: at least 3 min)")
print("All channels.tsv status 'good':", bool(rec_tab.status_all_good.all()))

Every recording is at 512 Hz, lasts 181 to 292 s (the paper says at least 3 min), and has the
32 scalp electrodes of the BioSemi 32 layout in the order of the montage, 8 EXG channels and a
Status channel. No electrode is marked bad. The two sessions that the `notes` column says were
exported from preprocessed EEGLAB files differ from the rest, and from each other:

- sub-pd6 `on`: electrode means are 0 and the 32 channels sum to 0.048 uV at most, so it
  already carries the common average reference; it lasts 289 s, against 184 to 216 s for the
  other `on` sessions. After preprocessing its electrode SDs (5.2 to 26.9 uV) are like the
  others'.
- sub-pd16 `on`: electrode means are small (23 uV, against thousands in raw files) but the
  channels do not sum to zero, and after the same preprocessing its electrode SDs are 11.5 to
  224 uV (median 62.6), about six times those of any other recording. The spectra below show
  it with about 100 times the power of the others and no alpha peak. The dataset does not say
  how this file was preprocessed or scaled.

Both have a high cutoff of 256 Hz in channels.tsv, where the others give 104 Hz. Sharpness,
steepness and PAC are ratios and do not depend on the signal's scale; group spectra do.
Results without these two patients are shown as a variant under the main claims.

In [ ]:
# events.tsv against the Status channel: the only time base in the files
rows = []
for i in info.itertuples():
    ev = side[(i.sub, i.ses)]["events"]
    trig = ev[ev.value != 65536]
    changes = [c for c in i.status_changes if c[1] not in (0, 65536)]
    rows.append({"sub": i.sub, "ses": i.ses, "events": "; ".join(f"{int(a)}:{int(b)}" for a, b in zip(ev["sample"], ev["value"])),
                 "status_first_nonzero_changes": "; ".join(f"{a}:{b}" for a, b in changes[:3])})
ev_tab = pd.DataFrame(rows)
print(ev_tab.to_string(index=False))

The events files hold one trigger per recording (value 1, 1 to 30 s into the file; sub-pd6
`on` at 95.6 s, sub-pd16 `on` an unexplained value), and the Status channel changes at the same
samples. The task is rest, with no trials or conditions in time, so there is no trial table
whose clock the EEG has to share; the paper uses the whole recording. What the trigger marks is
not described; the analysis uses the whole recording, as the claims file says.

In [ ]:
# recordings deposited twice: identical raw samples at the start of C3, across all pairs
heads = {(i.sub, i.ses): i.raw_c3_head for i in info.itertuples()}
dupes = []
for a, b in itertools.combinations(heads, 2):
    x, y = heads[a], heads[b]
    if np.allclose(x, y) or abs(np.corrcoef(x - x.mean(), y - y.mean())[0, 1]) > 0.99:
        dupes.append((a, b))
print("Pairs of recordings with identical (or r > 0.99) first 4 s of raw C3:", dupes)
print("Distinct BDF checksums:", bdf["id"].nunique(), "of", len(bdf))
cnt = shape[(shape.signal.isin(["C3", "C4"])) & (shape.band == "beta")]
print("\nBeta peaks per recording at C3 and C4: %d to %d (whole recordings, 100-sample edges dropped)"
      % (cnt.n_peaks.min(), cnt.n_peaks.max()))
cnt = shape[(shape.signal.isin(["C3", "C4"])) & (shape.band == "alpha")]
print("Alpha peaks per recording at C3 and C4: %d to %d" % (cnt.n_peaks.min(), cnt.n_peaks.max()))

In [ ]:
beh = {}
for r in recs.itertuples():
    beh[(r.sub, r.ses)] = ws.read_json(f"sub-{r.sub}/ses-{r.ses}/beh/sub-{r.sub}_ses-{r.ses}_task-rest_beh.json")
print("Root task-rest_beh.json:", ws.read_json("task-rest_beh.json"))
keys = sorted({k for v in beh.values() for k in v["questionairres"]})
print("Questionnaire fields in the beh JSON files:", keys)
q = pd.DataFrame([{"sub": s, "ses": e, **v["questionairres"]} for (s, e), v in beh.items()])
print(q[q.ses != "hc"].sort_values(["sub", "ses"]).to_string(index=False))
print("Controls' UPDRS values:", sorted(q[q.ses == "hc"]["UPDRS"].unique().tolist()))
pdq = q[q.ses != "hc"]
print("Total UPDRS minus UPDRS 18-26: %d to %d" % ((pdq["Total UPDRS"] - pdq["UPDRS 18-26"]).min(),
                                                   (pdq["Total UPDRS"] - pdq["UPDRS 18-26"]).max()))

The UPDRS fields have no description in the dataset (the only `beh` sidecar describes a dummy
`trial` column). "Total UPDRS" ranges from 16 to 58 and exceeds "UPDRS 18-26" by 0 to 14,
which fits the motor examination (part III, items 18 to 31) and its items 18 to 26; this is
consistent with the values, not shown.

### Tables used below

`composite` gives one value per participant and session: the mean of a measure over two
signals (C3 and C4 unless stated), as the paper's composite (Methods, Data preprocessing,
paragraph 2). Patients are paired by subject id.

In [ ]:
def composite(table, value, signals=("C3", "C4"), **sel):
    t = table[table.signal.isin(signals)]
    for k, v in sel.items():
        t = t[t[k] == v]
    out = t.groupby(["sub", "ses"])[value].mean().unstack("ses")
    assert (t.groupby(["sub", "ses"]).size() == len(signals)).all()
    return out


def paired(c):
    p = c[["off", "on"]].dropna()
    return p.loc[sorted(p.index, key=lambda s: int(s[2:]))]


def ctrl(c):
    return c["hc"].dropna()


def offon_row(c, label):
    p = paired(c)
    W, pa = ws.signed_rank(p.off, p.on)
    _, pe = ws.signed_rank(p.off, p.on, method="exact")
    return {"test": label, "n": len(p), "W": W, "p": pa, "p_exact": pe, "off>on": int((p.off > p.on).sum()),
            "d": ws.cohen_d(p.off, p.on), "mean_off": p.off.mean(), "mean_on": p.on.mean()}


def offctrl_row(c, label):
    p = paired(c)
    z, pz = ws.rank_sum(p.off, ctrl(c))
    return {"test": label, "n_off": len(p), "n_ctrl": len(ctrl(c)), "z": z, "p": pz,
            "d": ws.cohen_d(p.off, ctrl(c)), "mean_off": p.off.mean(), "mean_ctrl": ctrl(c).mean()}


SHR = lambda band="beta", taps="118", sig=("C3", "C4"): composite(shape, "ShR", sig, band=band, taps=taps)
STR = lambda band="beta", taps="118", sig=("C3", "C4"): composite(shape, "StR", sig, band=band, taps=taps)
PAC = lambda band="beta", reading="methods", sig=("C3", "C4"): composite(pac, "PAC", sig, band=band, reading=reading)
EXCL = {"pd6", "pd16"}   # on sessions exported from preprocessed EEGLAB files

# Table 1 (off against on) under every combination of the filter readings, FDR over its six tests
t1_rows = []
for a_taps, a_pac, b_pac in itertools.product(["192", "118"], ["a192", "b231", "c118"], ["methods", "code"]):
    tests = [offon_row(SHR("alpha", a_taps), "alpha ShR"), offon_row(STR("alpha", a_taps), "alpha StR"),
             offon_row(PAC("alpha", a_pac), "alpha PAC"), offon_row(SHR(), "beta ShR"),
             offon_row(STR(), "beta StR"), offon_row(PAC("beta", b_pac), "beta PAC")]
    q_ = ws.fdr_bh([t["p"] for t in tests])
    for t, qq in zip(tests, q_):
        t1_rows.append({"alpha_shape_taps": a_taps, "alpha_pac": a_pac, "beta_pac": b_pac, **t, "p_fdr": qq})
table1 = pd.DataFrame(t1_rows)

t2_rows = []
for a_taps, a_pac, b_pac in itertools.product(["192", "118"], ["a192", "b231", "c118"], ["methods", "code"]):
    tests = [offctrl_row(SHR("alpha", a_taps), "alpha ShR"), offctrl_row(STR("alpha", a_taps), "alpha StR"),
             offctrl_row(PAC("alpha", a_pac), "alpha PAC"), offctrl_row(SHR(), "beta ShR"),
             offctrl_row(STR(), "beta StR"), offctrl_row(PAC("beta", b_pac), "beta PAC")]
    q_ = ws.fdr_bh([t["p"] for t in tests])
    for t, qq in zip(tests, q_):
        t2_rows.append({"alpha_shape_taps": a_taps, "alpha_pac": a_pac, "beta_pac": b_pac, **t, "p_fdr": qq})
table2 = pd.DataFrame(t2_rows)
MAIN = (table1.alpha_shape_taps == "192") & (table1.alpha_pac == "a192") & (table1.beta_pac == "methods")
print("Table 1, off against on, main readings (alpha shape 192 taps, alpha PAC 192/123, beta PAC 118/123):")
print(table1[MAIN].drop(columns=["alpha_shape_taps", "alpha_pac", "beta_pac"]).round(4).to_string(index=False))
MAIN2 = (table2.alpha_shape_taps == "192") & (table2.alpha_pac == "a192") & (table2.beta_pac == "methods")
print("\nTable 2, off against controls, main readings:")
print(table2[MAIN2].drop(columns=["alpha_shape_taps", "alpha_pac", "beta_pac"]).round(4).to_string(index=False))


def fdr_range(table, test):
    t = table[table.test == test]
    return t.p_fdr.min(), t.p_fdr.max()


def show_patients(c, label):
    p = paired(c)
    print(f"{label} per patient (off, on, off - on):")
    print(pd.DataFrame({"off": p.off, "on": p.on, "off-on": p.off - p.on}).round(4).T.to_string())

## fig2-sharpness-off-on

> Over sensorimotor electrodes (C3 and C4 composite), beta sharpness ratio is higher in PD
> patients off medication than on medication. Expected: signed rank 12 (+/- 1.2), d 0.86
> (+/- 0.086), FDR corrected p < 0.05 over the six tests of Table 1.

In [ ]:
c = SHR()
show_patients(c, "Beta sharpness ratio (C3/C4 composite)")
r = offon_row(c, "beta ShR")
lo, hi = fdr_range(table1, "beta ShR")
print("\nSigned rank W = %.1f (expected 12 +/- 1.2), p = %.4f (normal approximation; exact %.4f), d = %.3f (expected 0.86 +/- 0.086)"
      % (r["W"], r["p"], r["p_exact"], r["d"]))
print("FDR corrected p over the six tests of Table 1: %.4f to %.4f across the 12 filter-reading combinations" % (lo, hi))
print("Off higher than on in %d of %d patients" % (r["off>on"], r["n"]))
ex = paired(c).drop(index=list(EXCL))
W, p = ws.signed_rank(ex.off, ex.on)
print("Variant without pd6 and pd16 (n = %d): W = %.1f, p = %.4f, d = %.3f" % (len(ex), W, p, ws.cohen_d(ex.off, ex.on)))
fig, ax = plt.subplots(figsize=(4, 4))
p_ = paired(c)
ax.scatter(p_.off, p_.on)
m = max(p_.max())
ax.plot([0, m], [0, m], "k-", lw=0.5)
ax.set_xlabel("sharpness ratio off")
ax.set_ylabel("sharpness ratio on")
plt.show()
print("Plotted range: off %.4f to %.4f, on %.4f to %.4f; no point outside the axes" % (p_.off.min(), p_.off.max(), p_.on.min(), p_.on.max()))

**Result: partially reproduced.** Sharpness ratio is higher off than on in 13 of 15 patients
(W = 13, within 12 +/- 1.2; p = 0.0076; FDR corrected p = 0.0228 under every filter
reading), but d = 0.73 is below the tolerance of 0.86 +/- 0.086. No artifact samples were
rejected, as the dataset holds no marks. Without pd6 and pd16, W = 11, p = 0.016, d = 0.81.

## fig2-steepness-off-on

> Over sensorimotor electrodes, beta steepness ratio is higher in PD patients off medication
> than on medication. Expected: signed rank 9 (+/- 0.9), d 0.68 (+/- 0.068).

In [ ]:
c = STR()
show_patients(c, "Beta steepness ratio (C3/C4 composite)")
r = offon_row(c, "beta StR")
lo, hi = fdr_range(table1, "beta StR")
print("\nSigned rank W = %.1f (expected 9 +/- 0.9), p = %.4f (exact %.4f), d = %.3f (expected 0.68 +/- 0.068)"
      % (r["W"], r["p"], r["p_exact"], r["d"]))
print("FDR corrected p over Table 1: %.4f to %.4f across the filter readings" % (lo, hi))
print("Off higher than on in %d of %d patients" % (r["off>on"], r["n"]))
ex = paired(c).drop(index=list(EXCL))
W, p = ws.signed_rank(ex.off, ex.on)
print("Variant without pd6 and pd16 (n = %d): W = %.1f, p = %.4f, d = %.3f" % (len(ex), W, p, ws.cohen_d(ex.off, ex.on)))

**Result: partially reproduced.** Steepness ratio is higher off than on in 12 of 15 patients
(p = 0.0125, FDR corrected 0.0249) and d = 0.70 is within 0.68 +/- 0.068, but W = 16 is
outside 9 +/- 0.9. Without pd6 and pd16, W = 5, p = 0.0046, d = 0.75.

## fig2-pac-off-on

> Over sensorimotor electrodes, beta-gamma PAC is higher in PD patients off medication than on
> medication. Expected: signed rank 15 (+/- 1.5), d 0.83 (+/- 0.083, Table 1).

In [ ]:
rows = []
for reading in ["methods", "code"]:
    c = PAC("beta", reading)
    r = offon_row(c, f"beta PAC, {reading} filters")
    p_ = paired(c)
    r["d_log10"] = ws.cohen_d(np.log10(p_.off), np.log10(p_.on))
    lo, hi = fdr_range(table1[table1.beta_pac == reading], "beta PAC")
    r["p_fdr_min"], r["p_fdr_max"] = lo, hi
    rows.append(r)
print(pd.DataFrame(rows).round(4).to_string(index=False))
show_patients(PAC("beta", "methods"), "Beta PAC, Methods filters (118/123 samples)")
print("Expected: W 15 +/- 1.5 (13.5 to 16.5); d 0.83 +/- 0.083 (0.747 to 0.913)")

**Result: partially reproduced.** PAC is higher off than on (p = 0.0076 with the Methods'
filters and 0.0064 with the code's, FDR corrected 0.0228), and d on raw PAC is 0.85 and 0.86,
within 0.83 +/- 0.083 (on log10 PAC 0.73 and 0.86). The signed rank statistic is 13 and 12,
outside 15 +/- 1.5 under both filter readings.

## bipolar-off-on

> With a bipolar montage (C3-CP1 and C4-CP2), sharpness ratio, steepness ratio and PAC each
> differ between PD patients off and on medication (paper: p < 0.004, FDR corrected for each).

In [ ]:
BIP = ("C3-CP1", "C4-CP2")
for reading in ["methods", "code"]:
    tests = [offon_row(SHR(sig=BIP), "bipolar ShR"), offon_row(STR(sig=BIP), "bipolar StR"),
             offon_row(PAC("beta", reading, BIP), f"bipolar PAC ({reading})")]
    t = pd.DataFrame(tests)
    t["p_fdr3"] = ws.fdr_bh(t.p)
    print(t.round(4).to_string(index=False), "\n")
print("Paper bound: p < 0.004 FDR corrected for each measure.")

**Result: reproduced.** With the bipolar pairs all three measures are higher off than on (13,
13 and 12 or 13 of 15 patients), and all three FDR corrected p-values are below 0.05 under
both PAC filter readings (largest 0.0106). None is below the paper's bound of 0.004 after
correction (steepness 0.0044, sharpness 0.0106, PAC 0.0096 or 0.0081); uncorrected, only
steepness is (0.0015).

## temporalis-f7f8-null and temporalis-t7t8-null

> Over F7 and F8 (and over T7 and T8), none of sharpness ratio, steepness ratio or PAC differs
> between patients off and on medication (paper F7/F8: p = 0.73, 0.394, 0.191 uncorrected).

In [ ]:
FT = ("F7", "F8")
rows = [offon_row(SHR(sig=FT), "F7/F8 ShR"), offon_row(STR(sig=FT), "F7/F8 StR"),
        offon_row(PAC("beta", "methods", FT), "F7/F8 PAC (methods)"), offon_row(PAC("beta", "code", FT), "F7/F8 PAC (code)")]
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Paper: sharpness p = 0.73, steepness p = 0.394, PAC p = 0.191, uncorrected")

**Result: reproduced.** At F7 and F8 no measure differs at p < 0.05 uncorrected: sharpness
ratio p = 0.69 (d 0.31), steepness ratio p = 0.17 (d 0.57), PAC p = 0.0995 with the Methods'
filters and 0.078 with the code's (d 0.67).

In [ ]:
TT = ("T7", "T8")
rows = [offon_row(SHR(sig=TT), "T7/T8 ShR"), offon_row(STR(sig=TT), "T7/T8 StR"),
        offon_row(PAC("beta", "methods", TT), "T7/T8 PAC (methods)"), offon_row(PAC("beta", "code", TT), "T7/T8 PAC (code)")]
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** At T7 and T8 no measure differs at p < 0.05 uncorrected (sharpness
p = 0.36, steepness p = 0.95, PAC p = 0.11 under both filter readings).

## fig2c-topography-sensorimotor

> Scalp topographies of the off against on medication test statistic for each measure are
> consistent with a sensorimotor origin. Criterion: for each beta measure, the electrode with
> the smallest p is C3, Cz, C4 or a neighbour (FC1, FC2, FC5, FC6, CP1, CP2, CP5, CP6), and none
> of F7, F8, T7, T8 is significant (p < 0.05 uncorrected, or FDR over electrodes).

In [ ]:
SENSORI = {"C3", "Cz", "C4", "FC1", "FC2", "FC5", "FC6", "CP1", "CP2", "CP5", "CP6"}
TEMPORAL = ["F7", "F8", "T7", "T8"]
topo = []
for label, tab, val, sel in [("ShR", shape, "ShR", {"band": "beta", "taps": "118"}),
                             ("StR", shape, "StR", {"band": "beta", "taps": "118"}),
                             ("PAC methods", pac, "PAC", {"band": "beta", "reading": "methods"}),
                             ("PAC code", pac, "PAC", {"band": "beta", "reading": "code"})]:
    t = tab[tab.signal.isin(ws.SCALP)]
    for k, v in sel.items():
        t = t[t[k] == v]
    wide = t.pivot_table(index=["sub", "ses"], columns="signal", values=val)
    for ch in ws.SCALP:
        cc = wide[ch].unstack("ses")
        p_ = paired(cc)
        W, p = ws.signed_rank(p_.off, p_.on)
        topo.append({"measure": label, "electrode": ch, "W": W, "p": p, "off>on": int((p_.off > p_.on).sum())})
topo = pd.DataFrame(topo)
topo["p_fdr32"] = topo.groupby("measure").p.transform(lambda s: ws.fdr_bh(s.values))
for m, t in topo.groupby("measure", sort=False):
    t = t.sort_values("p")
    best = t.iloc[0]
    tem = t[t.electrode.isin(TEMPORAL)]
    print(f"{m}: smallest p at {best.electrode} (p = {best.p:.4f}); in sensorimotor set: {best.electrode in SENSORI}; "
          f"significant uncorrected: {t[t.p < 0.05].electrode.tolist()}; FDR: {t[t.p_fdr32 < 0.05].electrode.tolist()}")
    print("   F7/F8/T7/T8 p:", dict(zip(tem.electrode, tem.p.round(4))), "\n   five smallest:",
          dict(zip(t.electrode[:5], t.p[:5].round(4))))
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, m in zip(axes, ["ShR", "StR", "PAC methods"]):
    t = topo[topo.measure == m].set_index("electrode").loc[ws.SCALP]
    ax.bar(range(32), -np.log10(t.p))
    ax.axhline(-np.log10(0.05), color="k", lw=0.5)
    ax.set_xticks(range(32), ws.SCALP, rotation=90, fontsize=6)
    ax.set_title(m)
    ax.set_ylabel("-log10 p")
plt.tight_layout()
plt.show()
print("-log10 p range plotted: %.2f to %.2f; nothing outside the axes" % ((-np.log10(topo.p)).min(), (-np.log10(topo.p)).max()))

**Result: partially reproduced.** Steepness ratio meets the criterion: the smallest p is at
C4 (0.023) and none of F7, F8, T7, T8 is significant. PAC meets it with the Methods' filters
under the FDR reading (smallest p at CP6, 0.027; F7 has p = 0.041 uncorrected, so not under
the uncorrected reading), but with the code's filters the smallest p is at Fz (0.020).
Sharpness ratio does not: its smallest p is at P8 (0.0064), then CP6 and AF3. No electrode
survives FDR correction over the 32 for any measure.

## offctrl-sharpness-null

> Beta sharpness ratio over sensorimotor electrodes does not differ between patients off
> medication and controls (paper: rank sum 0.91, p = 0.363; d 0.19).

In [ ]:
r = offctrl_row(SHR(), "beta ShR, off vs controls")
print(pd.DataFrame([r]).round(4).to_string(index=False))
print("Paper: z = 0.91, p = 0.3631 uncorrected, d = 0.19 (+/- 0.019)")

**Result: reproduced.** The rank sum test finds no difference (z = 0.47, p = 0.64). d = 0.09
is outside the expected 0.19 +/- 0.019; the claim of absence is judged on the test.

## offctrl-steepness

> Beta steepness ratio over sensorimotor electrodes is higher in patients off medication than
> in controls, with the average reference and with a bipolar montage. Expected: z 2.53
> (+/- 0.253), d 0.88 (+/- 0.088); FDR over Table 2 for the average reference; bipolar
> significant under one reading (FDR over the three bipolar measures, or the single test).

In [ ]:
r = offctrl_row(STR(), "beta StR, average reference")
lo, hi = fdr_range(table2, "beta StR")
print(pd.DataFrame([r]).round(4).to_string(index=False))
print("FDR corrected over the six tests of Table 2: %.4f to %.4f across filter readings" % (lo, hi))
for reading in ["methods", "code"]:
    b = [offctrl_row(SHR(sig=BIP), "bipolar ShR"), offctrl_row(STR(sig=BIP), "bipolar StR"),
         offctrl_row(PAC("beta", reading, BIP), f"bipolar PAC ({reading})")]
    b = pd.DataFrame(b)
    b["p_fdr3"] = ws.fdr_bh(b.p)
    print("\nBipolar montage, off against controls:")
    print(b.round(4).to_string(index=False))
print("\nPaper: average reference z = 2.53, p = 0.011 (FDR 0.0331), d = 0.88; bipolar p = 0.028 FDR corrected")

**Result: partially reproduced.** With the average reference steepness ratio is higher off
than in controls with z = 2.37 (within 2.53 +/- 0.253) and d = 0.85 (within 0.88 +/- 0.088),
p = 0.018 uncorrected, but FDR corrected over the six tests of Table 2 p = 0.0531, just above
0.05, under every filter reading. With the bipolar pairs z = 2.37, p = 0.018 as a single test
(significant) and 0.0531 FDR corrected over the three bipolar measures. The criterion does not
name this case (bipolar holds, average reference corrected test does not); the effect is
present with the paper's size but misses the corrected threshold.

## offctrl-pac-null and offctrl-pac-csd

> With the average reference, beta PAC does not differ between patients off and controls
> (paper p = 0.236). With a current source density montage, beta PAC is higher off than in
> controls (paper p = 0.022).

In [ ]:
rows = [offctrl_row(PAC("beta", rd), f"beta PAC average reference ({rd})") for rd in ["methods", "code"]]
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Paper: z = 1.19, p = 0.236 uncorrected")

**Result: reproduced.** No difference with the average reference: p = 0.32 with the Methods'
filters (z = 0.99) and 0.29 with the code's (z = 1.07).

In [ ]:
CSD = ("CSD C3", "CSD C4")
rows = [offctrl_row(PAC("beta", rd, CSD), f"beta PAC, CSD ({rd})") for rd in ["methods", "code"]]
print(pd.DataFrame(rows).round(4).to_string(index=False))
rows = [offon_row(PAC("beta", rd, CSD), f"beta PAC, CSD, off vs on ({rd})") for rd in ["methods", "code"]]
print("\nFor reference, CSD PAC off against on:")
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Paper: off against controls p = 0.022")

**Result: not reproduced.** With the CSD montage PAC is higher off than in controls (z = 1.78)
but not significantly (p = 0.075 under both filter readings), where the paper reports
p = 0.022.

## updrs-improvement

> UPDRS is lower on medication than off (paper: signed rank 3.5, p = 0.003). Readings: motor
> part (III) total and the total of all parts the dataset holds.

In [ ]:
u = q[q.ses != "hc"].pivot(index="sub", columns="ses")
rows = []
for field in ["Total UPDRS", "UPDRS 18-26"]:
    off, on = u[(field, "off")].astype(float), u[(field, "on")].astype(float)
    W, p = ws.signed_rank(off, on)
    rows.append({"field": field, "n": len(off), "W": W, "p": p, "median_off": off.median(), "median_on": on.median(),
                 "lower_on": int((on < off).sum()), "equal": int((on == off).sum()), "higher_on": int((on > off).sum())})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("scipy's signed rank (zero_method 'wilcox') drops tied pairs; with 'pratt' or 'zsplit':")
for field in ["Total UPDRS", "UPDRS 18-26"]:
    off, on = u[(field, "off")].astype(float), u[(field, "on")].astype(float)
    for zm in ["pratt", "zsplit"]:
        r_ = stats.wilcoxon(off, on, zero_method=zm, method="approx", correction=False)
        print(f"  {field}, {zm}: W = {r_.statistic:.1f}, p = {r_.pvalue:.4f}")
print("Paper: W = 3.5, p = 0.003; expected W 3.5 +/- 0.35")

**Result: reproduced.** "Total UPDRS", the only total in the dataset and by its values the
motor examination, is lower on medication in 12 patients, equal in 2 and higher in 1:
W = 3.5, p = 0.0033, the paper's values. Items 18 to 26 alone give W = 6.5, p = 0.011.

## fig1c-beta-alpha-peaks

> The average power spectrum of each group shows an oscillation in the beta band and a
> prominent mu/alpha oscillation. Criterion: a local maximum between 8 and 12 Hz and one between
> 13 and 30 Hz in each group's average Welch spectrum (C3 and C4; all electrodes a variant).

In [ ]:
freqs = results[0]["freqs"]
spec = {}
for r_, rec in zip(results, recs.itertuples()):
    spec[(rec.sub, rec.ses)] = r_["psd"]


def group_spectrum(ses, chans):
    return np.mean([spec[k].loc[list(chans)].values.mean(axis=0) for k in spec if k[1] == ses], axis=0)


def local_max(s, lo, hi):
    i = np.arange(1, len(s) - 1)
    peak = (s[i] > s[i - 1]) & (s[i] > s[i + 1]) & (freqs[i] >= lo) & (freqs[i] <= hi)
    return [round(float(f), 2) for f in freqs[i][peak]]


fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (label, chans) in zip(axes, [("C3 and C4", ["C3", "C4"]), ("all 32 electrodes", ws.SCALP)]):
    for ses in ["off", "on", "hc"]:
        s = group_spectrum(ses, chans)
        print(f"{label}, {GROUP[ses]}: local maxima 8-12 Hz {local_max(s, 8, 12)}, 13-30 Hz {local_max(s, 13, 30)}")
        keep = freqs <= 45
        ax.semilogy(freqs[keep], s[keep], label=GROUP[ses])
    ax.set_title(label)
    ax.set_xlabel("Hz")
    ax.set_ylabel("power (uV^2/Hz)")
    ax.legend()
plt.tight_layout()
plt.show()
print("Frequency resolution %.3f Hz; plotted 0 to 45 Hz" % (freqs[1] - freqs[0]))
print("\nVariants (not the criterion): geometric mean over recordings (mean of log10 power), and the arithmetic mean without sub-pd16 on")
for ses in ["off", "on", "hc"]:
    s_log = 10 ** np.mean([np.log10(spec[k].loc[["C3", "C4"]].values.mean(axis=0)) for k in spec if k[1] == ses], axis=0)
    s_ex = np.mean([spec[k].loc[["C3", "C4"]].values.mean(axis=0) for k in spec if k[1] == ses and k != ("pd16", "on")], axis=0)
    print(f"  {GROUP[ses]}: log mean 8-12 Hz {local_max(s_log, 8, 12)}, 13-30 Hz {local_max(s_log, 13, 30)}; "
          f"without pd16 on 8-12 Hz {local_max(s_ex, 8, 12)}, 13-30 Hz {local_max(s_ex, 13, 30)}; "
          f"local maxima 6-13 Hz of the mean {local_max(group_spectrum(ses, ['C3', 'C4']), 6, 13)}")
for ses in ["off", "on", "hc"]:
    n_alpha = sum(bool(local_max(spec[k].loc[["C3", "C4"]].values.mean(axis=0), 8, 12)) for k in spec if k[1] == ses)
    n_beta = sum(bool(local_max(spec[k].loc[["C3", "C4"]].values.mean(axis=0), 13, 30)) for k in spec if k[1] == ses)
    n = sum(1 for k in spec if k[1] == ses)
    print(f"Per recording, {GROUP[ses]}: alpha local max in {n_alpha}/{n}, beta local max in {n_beta}/{n}")

**Result: not reproduced.** In the average C3 and C4 spectrum only the controls have a local
maximum between 8 and 12 Hz (10.24 Hz) and between 13 and 30 Hz (18.2 Hz). The off group's
mean spectrum peaks at 7.96 Hz, just below the band, and the on group's has no peak from 6 to
30 Hz; the on mean is dominated by sub-pd16 `on`, but without it there is still none. Over all
electrodes the off group has beta maxima (17.1 and 21.6 Hz) and still no alpha maximum. Single
recordings have an alpha maximum in 8 of 15 (off), 8 of 15 (on) and 11 of 16 (controls). The
paper's spectra were computed after artifact rejection, which is not possible here.

## table1-alpha-shape-off-on

> In the mu/alpha band (8 to 12 Hz), sharpness ratio and steepness ratio are also higher off
> medication than on. Expected: sharpness W 20 (+/- 2), d 0.70 (+/- 0.07); steepness W 12
> (+/- 1.2), d 0.60 (+/- 0.06); FDR corrected p < 0.05 over Table 1.

In [ ]:
t = table1[table1.test.isin(["alpha ShR", "alpha StR"])]
print(t.groupby(["alpha_shape_taps", "test"]).agg(W=("W", "first"), p=("p", "first"), d=("d", "first"),
      off_gt_on=("off>on", "first"), p_fdr_min=("p_fdr", "min"), p_fdr_max=("p_fdr", "max")).round(4).to_string())
print("Expected: ShR W 18 to 22, d 0.63 to 0.77; StR W 10.8 to 13.2, d 0.54 to 0.66")

**Result: partially reproduced.** Both alpha measures are higher off than on with FDR corrected
p below 0.05 under both shape filter readings (largest 0.0321). Sharpness ratio matches
(W = 18 or 19, d = 0.71 or 0.66) and the steepness d matches (0.58 or 0.59), but the steepness
W = 21 is outside 12 +/- 1.2.

## table1-alpha-pac-null

> In the mu/alpha band, PAC does not differ significantly between patients off and on
> medication (FDR corrected over Table 1, at 0.05; paper corrected p = 0.0881, d 0.51).

In [ ]:
t = table1[table1.test == "alpha PAC"]
print(t.groupby("alpha_pac").agg(W=("W", "first"), p=("p", "first"), d=("d", "first"),
      p_fdr_min=("p_fdr", "min"), p_fdr_max=("p_fdr", "max")).round(4).to_string())
for rd in ["a192", "b231", "c118"]:
    p_ = paired(PAC("alpha", rd))
    print(f"{rd}: d on log10 PAC = {ws.cohen_d(np.log10(p_.off), np.log10(p_.on)):.3f}")
print("Readings: a192 = 192-tap phase filter (3 cycles of 8 Hz) with the Methods' 123-sample gamma filter; "
      "b231 = the code's 231/240 taps; c118 = the Methods' 118 samples with 123")

**Result: reproduced, resting on an assumption.** With the 192-tap phase filter (reading a)
alpha PAC does not differ after correction (W = 27, corrected p = 0.0609, d = 0.70), nor with
the code's 231/240 taps (reading b, corrected p = 0.0783); with the Methods' beta filter length
of 118 samples (reading c) it does (W = 23, corrected p = 0.0356). Reading a, pacpy's default
of three cycles of the low cutoff, is adopted as the assumption (see the summary). d is 0.70,
0.72 and 0.66, outside 0.51 +/- 0.051 under every reading; the claim of absence is judged on
the test.

## table2-alpha-offctrl

> In the mu/alpha band, steepness ratio is higher in patients off medication than in
> controls, while sharpness ratio and PAC do not differ after correction. Expected: steepness
> z 2.53 (+/- 0.253), d 0.87 (+/- 0.087).

In [ ]:
t = table2[table2.test.str.startswith("alpha")]
print(t.groupby(["test", "alpha_shape_taps", "alpha_pac"]).agg(z=("z", "first"), p=("p", "first"), d=("d", "first"),
      p_fdr_min=("p_fdr", "min"), p_fdr_max=("p_fdr", "max")).round(4).to_string())
print("Expected: StR z 2.277 to 2.783, d 0.783 to 0.957, corrected p < 0.05; ShR and PAC corrected p >= 0.05")

**Result: partially reproduced.** Alpha steepness ratio is higher off than in controls with
z = 2.45 (192 taps) or 2.53 (118 samples), within 2.53 +/- 0.253, and d = 0.89, within
0.87 +/- 0.087, but its FDR corrected p is 0.0531 under every reading, just above 0.05.
Sharpness ratio (corrected p 0.57 or 0.72) and PAC (0.12 to 0.41) do not differ after
correction, as stated.

## pac-beta-larger-than-alpha

> The off against on difference in PAC is larger in the beta band than in the mu/alpha band
> (paper: signed rank 11, p = 0.005). Per patient, composite PAC off minus on in beta and in
> alpha, signed rank test of the two differences; raw PAC main, log PAC a variant.

In [ ]:
rows = []
for b_rd, a_rd, scale in itertools.product(["methods", "code"], ["a192", "b231", "c118"], ["raw", "log10"]):
    pb, pa = paired(PAC("beta", b_rd)), paired(PAC("alpha", a_rd))
    f = (lambda v: v) if scale == "raw" else np.log10
    db, da = f(pb.off) - f(pb.on), f(pa.off) - f(pa.on)
    W, p = ws.signed_rank(db, da)
    rows.append({"beta": b_rd, "alpha": a_rd, "scale": scale, "W": W, "p": p,
                 "beta_diff_larger": int((db > da).sum()), "median_beta_diff": db.median(), "median_alpha_diff": da.median()})
print(pd.DataFrame(rows).round(5).to_string(index=False))
print("Expected: W 11 +/- 1.1 (9.9 to 12.1), p < 0.05, beta difference larger")

**Result: partially reproduced.** On raw PAC (the main reading) the beta off-on difference is
larger than the alpha one in 12 or 13 of 15 patients, p = 0.015 to 0.017 under every filter
reading, but W = 17 or 18 is outside 11 +/- 1.1. On log10 PAC (variant) there is no
difference (p = 0.26 to 0.73).

## fig3-sharpness-pac-correlation and fig3-steepness-pac-correlation

> Across patients, beta sharpness ratio (steepness ratio) and PAC over sensorimotor electrodes
> are positively correlated both off and on medication. Expected Spearman r: sharpness 0.70
> off, 0.74 on; steepness 0.74 off, 0.68 on (each +/- 10%).

In [ ]:
def corr_table(measure):
    rows = []
    for rd in ["methods", "code"]:
        cp, cm = paired(PAC("beta", rd)), paired(measure())
        for ses in ["off", "on"]:
            r_, p_ = stats.spearmanr(cm[ses], cp[ses])
            rows.append({"PAC filters": rd, "state": ses, "n": len(cm), "spearman_r": r_, "p": p_})
    return pd.DataFrame(rows)


print(corr_table(SHR).round(4).to_string(index=False))
print("Expected: off 0.70 (0.63 to 0.77), on 0.74 (0.666 to 0.814)")
cm, cp = paired(SHR()), paired(PAC("beta", "methods"))
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, ses in zip(axes, ["off", "on"]):
    ax.scatter(cm[ses], cp[ses])
    ax.set_xlabel(f"sharpness ratio {ses}")
    ax.set_ylabel(f"PAC {ses}")
plt.tight_layout()
plt.show()
print("Plotted ranges: sharpness %.4f to %.4f, PAC %.4f to %.4f" % (cm.min().min(), cm.max().max(), cp.min().min(), cp.max().max()))

**Result: partially reproduced.** Sharpness ratio and PAC are positively and significantly
correlated off and on under both PAC filter readings (r = 0.59 to 0.84, p = 0.0001 to
0.022). No reading has both r within tolerance: with the Methods' filters r = 0.59 off is
below 0.63 to 0.77 (on 0.79, within); with the code's, r = 0.84 on is above 0.666 to 0.814
(off 0.63, within).

In [ ]:
print(corr_table(STR).round(4).to_string(index=False))
print("Expected: off 0.74 (0.666 to 0.814), on 0.68 (0.612 to 0.748)")

**Result: partially reproduced.** Off medication steepness ratio and PAC are correlated as
reported (r = 0.80, p = 0.0003 with the Methods' filters; 0.80, p = 0.0004 with the code's).
On medication the correlation is weaker: r = 0.45, p = 0.095 with the Methods' filters, and
r = 0.53, p = 0.041 with the code's, below 0.612 to 0.748.

## fig4a-sensorimotor-q4

> Over sensorimotor electrodes, patients' beta waveforms mostly have sharper peaks than troughs
> and steeper decays than rises (Q4: log10 peak-to-trough ratio > 0 and log10 rise-to-fall
> ratio < 0). One point per patient, electrode (C3, C4) and state: 60 points.

In [ ]:
def quadrant(ptr, rdr):
    return np.select([(ptr > 0) & (rdr > 0), (ptr < 0) & (rdr > 0), (ptr < 0) & (rdr < 0), (ptr > 0) & (rdr < 0)],
                     ["Q1", "Q2", "Q3", "Q4"], "on an axis")


pts = shape[(shape.band == "beta") & (shape.taps == "118") & shape.signal.isin(ws.SCALP)].copy()
pts["quadrant"] = quadrant(pts.PTR.values, pts.RDR.values)
pts["group"] = pts.ses.map(GROUP)


def quad_counts(signals, groups):
    t = pts[pts.signal.isin(signals) & pts.group.isin(groups)]
    return t, t.quadrant.value_counts().reindex(["Q1", "Q2", "Q3", "Q4"], fill_value=0)


t, qc = quad_counts(["C3", "C4"], ["off", "on"])
print(f"Patients, C3 and C4, off and on: {len(t)} points")
print(qc.to_string(), "\nshare in Q4: %.3f" % (qc["Q4"] / len(t)))
print("\nBy electrode and state:")
print(pd.crosstab([t.signal, t.ses], t.quadrant).to_string())
print("\nBy patient (number of the 4 points in Q4):")
print(t.groupby("sub").quadrant.apply(lambda s: int((s == "Q4").sum())).to_string())
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, sig in zip(axes, [["C3", "C4"], ["F7", "F8"]]):
    tt = pts[pts.signal.isin(sig) & pts.group.isin(["off", "on"])]
    for ses, mk in [("off", "o"), ("on", "s")]:
        u_ = tt[tt.ses == ses]
        ax.scatter(u_.PTR, u_.RDR, marker=mk, label=ses)
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    ax.set_xlim(-0.3, 0.3)
    ax.set_ylim(-0.3, 0.3)
    ax.set_xlabel("log10 peak-to-trough ratio")
    ax.set_ylabel("log10 rise-to-fall ratio")
    ax.set_title("/".join(sig))
    ax.legend()
    out = ((tt.PTR.abs() > 0.3) | (tt.RDR.abs() > 0.3)).sum()
    print(f"{'/'.join(sig)}: PTR {tt.PTR.min():.3f} to {tt.PTR.max():.3f}, RDR {tt.RDR.min():.3f} to {tt.RDR.max():.3f}; {out} points outside the axes")
plt.tight_layout()
plt.show()

**Result: reproduced.** 48 of the 60 patient points at C3 and C4 (80%) are in Q4; Q1 holds 6,
Q2 4 and Q3 2.

## q4-patient-counts

> Nine of 15 patients have both off and on data in Q4 at C3, eight at C4; eight have both
> electrodes in Q4, giving nine patients with at least one electrode in Q4 on and off.

In [ ]:
pq = pts[pts.signal.isin(["C3", "C4"]) & pts.group.isin(["off", "on"])]
both = pq.assign(q4=pq.quadrant == "Q4").pivot_table(index="sub", columns=["signal", "ses"], values="q4", aggfunc="first")
meets = pd.DataFrame({"C3": both[("C3", "off")] & both[("C3", "on")], "C4": both[("C4", "off")] & both[("C4", "on")]})
print(meets.astype(int).T.to_string())
n_c3, n_c4 = int(meets.C3.sum()), int(meets.C4.sum())
n_both, n_any = int((meets.C3 & meets.C4).sum()), int((meets.C3 | meets.C4).sum())
print(f"\nPatients meeting the criterion: C3 {n_c3}, C4 {n_c4}, both {n_both}, at least one {n_any}")
print("Expected (+/- 1): C3 9, C4 8, both 8, at least one 9; swapped electrode order C3 8, C4 9")

**Result: not reproduced.** 10 patients have both off and on in Q4 at C3 and 10 at C4
(expected 9 and 8 +/- 1), 7 at both (expected 8 +/- 1) and 13 at one or more (expected
9 +/- 1). The C4 count and the count of patients with at least one electrode are outside
tolerance in either electrode order.

## fig4b-temporalis-variable-shape

> Electrodes closest to the temporalis muscles (F7 and F8) show a more variable waveform shape
> than sensorimotor electrodes. Readings: (a) a lower share of points in the most common
> quadrant at F7/F8; (b) a larger spread of the points about their centroid.

In [ ]:
rows = []
for sig in [["C3", "C4"], ["F7", "F8"]]:
    tt, qc = quad_counts(sig, ["off", "on"])
    xy = tt[["PTR", "RDR"]].values
    spread = np.sqrt(((xy - xy.mean(axis=0)) ** 2).sum(axis=1).mean())
    rows.append({"electrodes": "/".join(sig), "n": len(tt), **qc.to_dict(), "most_common": qc.idxmax(),
                 "share_most_common": qc.max() / len(tt), "rms_distance_to_centroid": spread,
                 "sd_PTR": tt.PTR.std(), "sd_RDR": tt.RDR.std()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** Both readings agree: at F7 and F8 the most common quadrant holds 68%
of the points (41 of 60, in Q2) against 80% (48 of 60, Q4) at C3 and C4, and the RMS distance
to the centroid is 0.093 against 0.076. The F7/F8 points are mostly in one other quadrant,
Q2 (sharper troughs, steeper rises), which reads as a different dominant shape more than an
absence of one.

## controls-q4

> Control participants show the same pattern, their sensorimotor beta waveforms falling mainly
> in Q4 (32 points: 16 controls, C3 and C4).

In [ ]:
t, qc = quad_counts(["C3", "C4"], ["control"])
print(f"Controls, C3 and C4: {len(t)} points")
print(qc.to_string(), "\nshare in Q4: %.3f" % (qc["Q4"] / len(t)))
print(pd.crosstab(t.signal, t.quadrant).to_string())

**Result: reproduced.** 25 of the 32 control points (78%) are in Q4 (13 of 16 at C3, 12 of 16
at C4).

## table3-q4-effect-size

> Restricting to patients whose sensorimotor waveforms fall in Q4 both off and on increases the
> off against on effect size of all three measures. Expected Q4 d: sharpness 0.95, steepness
> 0.70, PAC 0.83 (each +/- 10%); all-patient PAC d in Table 3 0.75.

In [ ]:
def q4_values(table, value, **sel):
    t = table[table.signal.isin(["C3", "C4"])]
    for k, v in sel.items():
        t = t[t[k] == v]
    w_ = t.pivot_table(index="sub", columns=["signal", "ses"], values=value)
    out = {}
    for s in meets.index:
        el = [e for e in ["C3", "C4"] if meets.loc[s, e]]
        if el:
            out[s] = {ses: np.mean([w_.loc[s, (e, ses)] for e in el]) for ses in ["off", "on"]}
    return pd.DataFrame(out).T


rows = []
for label, table, value, sel, f in [
        ("ShR", shape, "ShR", {"band": "beta", "taps": "118"}, None),
        ("StR", shape, "StR", {"band": "beta", "taps": "118"}, None),
        ("PAC raw, methods", pac, "PAC", {"band": "beta", "reading": "methods"}, None),
        ("PAC raw, code", pac, "PAC", {"band": "beta", "reading": "code"}, None),
        ("PAC log10, methods", pac, "PAC", {"band": "beta", "reading": "methods"}, np.log10),
        ("PAC log10, code", pac, "PAC", {"band": "beta", "reading": "code"}, np.log10)]:
    allp = paired(composite(table, value, **sel))
    q4 = q4_values(table, value, **sel)
    if f is not None:
        allp, q4 = f(allp), f(q4)
    rows.append({"measure": label, "n_all": len(allp), "d_all": ws.cohen_d(allp.off, allp.on),
                 "n_q4": len(q4), "d_q4": ws.cohen_d(q4.off, q4.on)})
t3 = pd.DataFrame(rows)
t3["increase"] = t3.d_q4 > t3.d_all
print(t3.round(3).to_string(index=False))
print("Expected Q4 d: ShR 0.95 (0.855 to 1.045), StR 0.70 (0.63 to 0.77), PAC 0.83 (0.747 to 0.913); all-patient PAC 0.75 (0.675 to 0.825)")

**Result: partially reproduced.** Restricting to the 13 patients with at least one electrode
in Q4 off and on raises d for all three measures under every reading (sharpness 0.73 to 0.99,
steepness 0.70 to 0.82, raw PAC 0.85 to 0.95 with the Methods' filters). Sharpness d (0.99) is
within 0.95 +/- 0.095; steepness (0.82) and PAC (0.95 to 1.07 across readings) are above
tolerance. The paper's 9 Q4 patients are 13 here (q4-patient-counts). The all-patient PAC d of
Table 3 (0.75) matches log10 PAC with the Methods' filters (0.73), as the claims file
suggests.

## Summary

| Claim | Outcome |
|---|---|
| fig2-sharpness-off-on | partially reproduced (W 13 matches, d 0.73 below 0.86) |
| fig2-steepness-off-on | partially reproduced (d 0.70 matches, W 16 against 9) |
| fig2-pac-off-on | partially reproduced (d 0.85 matches, W 13 against 15) |
| bipolar-off-on | reproduced |
| temporalis-f7f8-null | reproduced |
| temporalis-t7t8-null | reproduced |
| fig2c-topography-sensorimotor | partially reproduced (steepness and PAC, not sharpness) |
| offctrl-sharpness-null | reproduced |
| offctrl-steepness | partially reproduced (z and d match, FDR p 0.053) |
| offctrl-pac-null | reproduced |
| offctrl-pac-csd | not reproduced (p 0.075) |
| updrs-improvement | reproduced (W 3.5, p 0.0033) |
| fig1c-beta-alpha-peaks | not reproduced (only controls have both peaks) |
| table1-alpha-shape-off-on | partially reproduced (steepness W 21 against 12) |
| table1-alpha-pac-null | reproduced, resting on an assumption (alpha phase filter of 192 taps) |
| table2-alpha-offctrl | partially reproduced (steepness z and d match, FDR p 0.053) |
| pac-beta-larger-than-alpha | partially reproduced (p 0.015, W 17 against 11) |
| fig3-sharpness-pac-correlation | partially reproduced (one r out of tolerance) |
| fig3-steepness-pac-correlation | partially reproduced (on medication r 0.45 to 0.53) |
| fig4a-sensorimotor-q4 | reproduced (80% in Q4) |
| q4-patient-counts | not reproduced (13 patients against 9) |
| fig4b-temporalis-variable-shape | reproduced |
| controls-q4 | reproduced (78% in Q4) |
| table3-q4-effect-size | partially reproduced (d rises for all three; two values above tolerance) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The dataset holds neither the paper's manual
artifact marks nor its list of excluded electrodes, so no sample is rejected and all 32 scalp
electrodes enter the common average (readings (b) of the claims file). The paper's measures
were taken on its preprocessed MATLAB files, which are not public. Filter lengths the paper
leaves open are run under each reading the claims file names. The CSD uses MNE's spherical
spline with the CSD toolbox's parameters on template electrode positions.

**Assumption.** For alpha PAC the phase filter length is not given. The readings disagree on
table1-alpha-pac-null; the 192-tap filter (three cycles of 8 Hz) is adopted, as the default of
pacpy, the package the Methods name for PAC, and the rule the paper's code uses for its shape
filters. Two of the three readings give the same outcome.

**What was found about the dataset.** It holds the 15 patients (8 female, mean age 63.3) and 16
controls (9 female, 63.5) the paper describes, each recording once, with UPDRS scores that
give the paper's UPDRS test exactly (W = 3.5, p = 0.0033). The artifact marks and excluded
electrodes of the paper's preprocessing are not in it. Two `on` sessions (sub-pd6, sub-pd16)
were exported from preprocessed EEGLAB files; sub-pd16 `on` has about six times the amplitude
of any other recording and no alpha peak, and the dataset does not say how it was processed.
The UPDRS fields carry no description. Without artifact rejection the main off against on
results hold, with statistics close to the paper's.